# Beyond the Baseline — Eksperimen Final (A-NRO dan remedi imbalance lainnya)

Notebook ini mereproduksi **seluruh eksperimen paper 1** di lingkungan yang sama dengan baseline IKADO
(scikit-learn 1.6.1) dan memakai **imbalanced-learn** untuk SMOTE, SMOTE-NC, Borderline-SMOTE, dan ADASYN.

**Cara menjalankan**
1. Jalankan **Sel 1**. Kalau versi pustaka belum sesuai, sel ini memasang versi yang dikunci lalu **me-restart runtime otomatis** (muncul pesan "session crashed", itu normal).
2. Setelah restart, jalankan **Sel 1 sekali lagi** (harus muncul ✓ versi sesuai), lalu lanjutkan **Sel 2 dan seterusnya** secara berurutan (*Runtime → Run after*).
3. Di Sel 3, unggah file survei (`msme_digital_readiness_anonymized.csv`, tersedia dari penulis korespondensi atas permintaan).
4. Total waktu ±60–90 menit (CPU Colab biasa). Hasil per metode disimpan bertahap, jadi kalau koneksi putus cukup jalankan ulang: metode yang sudah selesai akan dilewati.
5. Hasil akhir: **`ANRO_results_final.xlsx`** dan **`ANRO_raw_results.zip`** (diunduh otomatis di sel terakhir). Kirim keduanya kembali.

**Protokol** (sama dengan IKADO): 19 item non-label + 7 karakteristik, imputasi dan one-hot di dalam fold,
repeated stratified 5-fold × 10 (seed 42), RF 500 pohon (seed 42). Resampling hanya pada fold latih.

In [ ]:
# SEL 1 — Kunci versi pustaka (auto-restart bila perlu)
import importlib, os, sys
REQ = {"scikit-learn": "1.6.1", "imbalanced-learn": "0.13.0"}
def _ver(mod):
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return None
ok = _ver("sklearn") == REQ["scikit-learn"] and _ver("imblearn") == REQ["imbalanced-learn"]
if not ok:
    print("Memasang versi yang dikunci ... runtime akan restart otomatis. Setelah itu jalankan Sel 1 lagi.")
    !pip install -q scikit-learn==1.6.1 imbalanced-learn==0.13.0
    os.kill(os.getpid(), 9)
else:
    import numpy, pandas, scipy, sklearn, imblearn, joblib
    print("✓ versi sesuai")
    for m in (sys, numpy, pandas, scipy, sklearn, imblearn, joblib):
        print(f"  {getattr(m, '__name__', 'python'):10s}", getattr(m, "__version__", sys.version.split()[0]))

In [ ]:
# SEL 2 — (opsional, disarankan) simpan hasil ke Google Drive agar aman bila koneksi putus
USE_DRIVE = True
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = "/content/drive/MyDrive/ANRO_results_v2"
else:
    RESULTS_DIR = "/content/results_v2"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Hasil disimpan di:", RESULTS_DIR)

In [ ]:
# SEL 3 — Unggah file survei (CSV anonim atau ekspor asli .xlsx); file ini tidak dipublikasikan
import os, glob
from google.colab import files
found = glob.glob("/content/*.csv") + glob.glob("/content/*Kesiapan*Digital*UMKM*.xlsx")
if found:
    DATA_PATH = found[0]
else:
    up = files.upload(); DATA_PATH = "/content/" + list(up.keys())[0]
os.environ["ANRO_DATA"] = DATA_PATH
print("Data:", DATA_PATH)


### Sel 4–9 — Menulis modul (encoding, MC-NRO/A-NRO, eksperimen, analisis, uji)

In [ ]:
%%writefile encode.py
"""Questionnaire encoding, identical to the final IKADO notebook (RF-MSME_final_revisi).

Every ordinal item is scored 1..5 following the option order of the Google Form.
Q20 follows the questionnaire ("Selalu Dibantu" = 1 ... "Tidak pernah dibantu" = 5),
Q24 is coded so that higher = greater concern (as in the IKADO paper).
Label items Q8, Q9, Q10, Q12, Q14 -> DRS (5-25) -> Low 5-11 / Medium 12-18 / High 19-25.
"""
import pandas as pd

LABEL_ITEMS = ["Q8", "Q9", "Q10", "Q12", "Q14"]
# Feature order reproduces the IKADO notebook exactly: Q2 sits at the END of the
# numeric block (it was dropped as "PII" because of the word "Smartphone" and
# re-appended later). Random Forest results depend on feature order, so this
# order is required to reproduce the published baseline (macro-F1 0.4470).
NONLABEL = [f"Q{i}" for i in range(1, 25) if f"Q{i}" not in LABEL_ITEMS + ["Q2"]] + ["Q2"]
DEMO = ["gender", "edu", "age", "btype", "market", "tenure", "turnover"]
DEMO_HEADERS = ["Jenis Kelamin", "Pendidikan Terakhir", "Usia", "Jenis Usaha/dagangan",
                "Lokasi Pasar", "Lama Jualan", "RataRata Omzet per BUlan"]

_FREQ = ["tidak pernah", "jarang", "kadang", "sering", "sangat sering"]
_MAMPU = ["sangat tidak mampu", "tidak mampu", "cukup mampu", "mampu", "sangat mampu"]
OPTIONS = {
    1: ["sangat tidak baik", "tidak baik", "cukup", "baik", "sangat baik"],
    2: ["sangat tidak memadai", "tidak memadai", "cukup memadai", "memadai", "sangat memadai"],
    3: ["tidak ada", "ada, tapi rusak", "ada, milik pribadi", "ada, milik usaha", "ada, lebih dari satu"],
    4: ["tidak pernah", "1- 3", "4 - 6", "7 -", "> 10"],
    5: _MAMPU, 6: _MAMPU, 8: _MAMPU,
    7: ["sangat tidak paham", "tidak paham", "cukup paham", "paham", "sangat paham"],
    9: ["hampir tidak ada", "hanya melayani 1-2", "melayani sebagian kecil",
        "melayani sebagian besar", "hampir seluruh"],
    10: ["tidak mempromosikan", "jarang", "kadang", "sering", "sangat sering"],
    11: _FREQ,
    12: ["tidak pernah", "sangat jarang", "kadang", "sering", "sangat sering"],
    13: ["sangat tidak paham", "kurang paham", "cukup paham", "paham", "sangat paham"],
    14: ["tidak pernah", "jarang", "kadang", "rutin", "sangat rutin"],
    15: ["sangat tidak penting", "kurang penting", "cukup penting", "penting", "sangat penting"],
    16: ["sama sekali", "manfaat kecil", "cukup bermanfaat", "bermanfaat", "sangat bermanfaat"],
    17: _FREQ, 18: _FREQ,
    19: ["sangat tidak mendukung", "kurang mendukung", "cukup mendukung", "mendukung", "sangat mendukung"],
    20: ["selalu dibantu", "sering dibantu", "kadang", "jarang dibantu", "tidak pernah dibantu"],
    21: ["sangat tidak percaya", "tidak percaya", "cukup percaya", "percaya", "sangat percaya"],
    22: ["sangat tidak aman", "tidak aman", "cukup aman", "aman", "sangat aman"],
    23: ["sangat tidak percaya", "kurang percaya", "cukup percaya", "percaya", "sangat percaya"],
    24: ["sangat tidak khawatir", "tidak khawatir", "cukup khawatir", "khawatir", "sangat khawatir"],
}


def _score(value, opts):
    """Longest matching option prefix -> score 1..5 (raises if unmatched)."""
    s = " ".join(str(value).replace(" ", " ").split()).lower()
    best = None
    for i, o in enumerate(opts):
        if s.startswith(o) and (best is None or len(o) > len(opts[best])):
            best = i
    if best is None:
        raise ValueError(f"Unmapped answer: {value!r}")
    return best + 1


def load(path):
    """Returns X (Q1..Q24 scores), demo (7 categorical columns), drs, y (0/1/2)."""
    d = pd.read_excel(path) if str(path).lower().endswith(".xlsx") else pd.read_csv(path)
    import re
    qcols = [c for c in d.columns if re.match(r"^\s*\d+\.", str(c))][:24]
    assert len(qcols) == 24, "Expected 24 questionnaire columns (1. ... 24.)"
    X = pd.DataFrame({f"Q{i+1}": d[c].map(lambda v, i=i: _score(v, OPTIONS[i + 1]))
                      for i, c in enumerate(qcols)})
    if all(h in d.columns for h in DEMO_HEADERS):      # raw export or anonymized CSV
        demo = d[DEMO_HEADERS].copy()
    else:                                               # fallback: original column positions
        demo = d.iloc[:, 4:11].copy()
    demo.columns = DEMO
    demo = demo.apply(lambda s: s.astype(str).str.strip())
    drs = X[LABEL_ITEMS].sum(axis=1)
    y = pd.cut(drs, [4, 11, 18, 25], labels=[0, 1, 2]).astype(int)
    return X, demo, drs, y


In [ ]:
%%writefile mcnro.py
"""
mcnro.py - Clean re-implementation of MC-NRO
(Multiclass Neighborhood Repartition-based Oversampling)

Reference
---------
S. Shen, Z. Li, Z. Huan, F. Shang, Y. Wang, Y. Chen (2024).
"Neighborhood repartition-based oversampling algorithm for multiclass
imbalanced data with label noise." Neurocomputing 600, 128090.
doi:10.1016/j.neucom.2024.128090

This module follows the PAPER (Algorithm 1 = B-NRO, Algorithm 2 = MC-NRO,
Eqs. 1-3), not the authors' public code, because the public code
(github.com/sshiyi77/MC-NRO) departs from the paper in several places.
Every place where the paper is ambiguous is marked "CHOICE" below, and the
differences from the public code are marked "CODE-DIFF".

Summary of CODE-DIFF (paper vs. public code)
  1. Distance: paper uses the L1 norm; code uses L2 everywhere
     (scipy distance_matrix is always L2).
  2. Radius: paper uses the distance to the k1-th majority neighbour;
     code uses argsort()[k1] - 0.01, i.e. the (k1+1)-th neighbour.
  3. First cleaning: with the default 'translate and remove' the code never
     applies the translations to the majority points (neither branch of the
     if/elif fires), so majority points stay where they were.
  4. Second cleaning: code matches moved points with np.in1d on flattened
     arrays and then deletes those *flat* indices as *row* indices, so it
     removes essentially arbitrary rows. It also uses majority_labels[0] as
     the label of every moved point, which is wrong in the multiclass case,
     and runs the cleaning inside every binary step instead of once after
     all classes have been oversampled (paper, Algorithm 2 lines 17-22).
  5. Potential: paper computes it w.r.t. the minority points inside the
     current neighbourhood; code uses all minority points.
  6. Repartition: paper uses R = phi0 + (max(U) - phi0)/n (Eq. 3); code uses
     fixed 0.33 thresholds borrowed from RBO/CCR ('D','S','E' regions),
     which corresponds to n = 3 with a slightly different max term.
  7. Candidate sampling: paper says uniform in the neighbourhood; code draws
     a normalised random cube direction times U(0,1)*r, which is NOT uniform
     in the ball (in 19 dimensions ~50% of candidates fall inside r/2,
     versus ~2e-6 for a uniform ball).
  8. IDF-weighted one-hot (datasets.py) is undone by the MinMaxScaler that is
     applied afterwards, so the weighting has no effect in their experiments.

Optional extension (NOT part of MC-NRO, off by default)
  `ordinal_cols` / `onehot_groups` project synthetic samples back to valid
  questionnaire values (round + clip ordinal items, arg-max for one-hot
  groups). This is the kind of "adaptive" step proposed for the MSME paper
  and must be reported as the authors' own modification.

Usage
-----
    from mcnro import MCNRO
    X_res, y_res = MCNRO(k1=5, k2=5, eps=0.5, n_partitions=3,
                         random_state=0).fit_resample(X_train, y_train)

Apply it to the TRAINING fold only (never before splitting).
"""
from __future__ import annotations

import numpy as np
from scipy.spatial.distance import cdist

__all__ = ["MCNRO", "ANRO", "BNRO"]


# ----------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------
def _cdist(A, B, p):
    if p == 1:
        return cdist(A, B, metric="cityblock")
    if p == 2:
        return cdist(A, B, metric="euclidean")
    return cdist(A, B, metric="minkowski", p=p)


def _uniform_in_ball(center, radius, size, p, rng):
    """Draw `size` points uniformly inside the Lp ball (p in {1, 2})."""
    m = center.shape[0]
    if p == 1:
        z = rng.laplace(size=(size, m))           # normalised -> uniform on L1 sphere
        norms = np.abs(z).sum(1, keepdims=True)
    elif p == 2:
        z = rng.standard_normal((size, m))        # normalised -> uniform on L2 sphere
        norms = np.sqrt((z ** 2).sum(1, keepdims=True))
    else:
        raise ValueError("uniform sampling implemented for p in {1, 2}")
    norms[norms == 0] = 1.0
    u = rng.random((size, 1)) ** (1.0 / m)        # radial part for uniform volume
    return center + z / norms * u * radius


def _potential(points, ref, eps, p):
    """Eq. 2: phi(x, C, eps) = sum_c exp(-(||c - x||_p / eps)^2)."""
    d = _cdist(np.atleast_2d(points), ref, p)
    return np.exp(-(d / eps) ** 2).sum(1)


def _log_potential(points, ref, eps, p):
    """log of Eq. 2 computed with log-sum-exp (used by A-NRO for ranking)."""
    from scipy.special import logsumexp
    d = _cdist(np.atleast_2d(points), ref, p)
    return logsumexp(-(d / eps) ** 2, axis=1)


def _allocate(radii, n_total):
    """Eq. 1: num_i proportional to 1/r_i.
    CHOICE: the paper floors each share, which may leave the class slightly
    below balance; the remainder is handed out by largest fractional part so
    the total equals n_total exactly (set exact=False in BNRO for pure floor)."""
    w = 1.0 / radii
    raw = w / w.sum() * n_total
    base = np.floor(raw).astype(int)
    return raw, base


# ----------------------------------------------------------------------------
# Algorithm 1 (Steps 1-2): binary NRO, used inside MC-NRO
# ----------------------------------------------------------------------------
class BNRO:
    """Binary NRO, Steps 1-2 of Algorithm 1 (Step 3 is done by MCNRO)."""

    def __init__(self, k1=5, eps=0.5, n_partitions=3, n_candidates=1000,
                 region="high", p=1, translate=True, exact=True,
                 fill="replace", repartition="paper", rng=None):
        assert region in ("high", "low")
        assert fill in ("replace", "strict")
        assert repartition in ("paper", "quantile")
        self.repartition = repartition
        self.k1, self.eps, self.n_partitions = k1, eps, n_partitions
        self.n_candidates, self.region, self.p = n_candidates, region, p
        self.translate, self.exact, self.fill = translate, exact, fill
        self.rng = rng if rng is not None else np.random.default_rng()

    def run(self, X_maj, X_min, n_generate):
        """Returns (X_maj_after, moved_mask, X_syn, info)."""
        rng, p = self.rng, self.p
        X_maj = X_maj.astype(float).copy()
        n_min, n_maj = len(X_min), len(X_maj)
        k1 = int(min(self.k1, n_maj))                  # guard for tiny folds
        D = _cdist(X_min, X_maj, p)

        # ---- Step 1: neighbourhood radius = distance to k1-th majority NN
        radii = np.sort(D, axis=1)[:, k1 - 1].copy()
        tiny = 1e-9
        radii[radii < tiny] = tiny                     # CHOICE: duplicates across classes

        # ---- Step 1: move majority points that lie inside each neighbourhood
        #      (Alg. 1 lines 12-15):  t_j += (r_i - d_ij)/d_ij * (x_j - x_i)
        T = np.zeros_like(X_maj)
        for i in range(n_min):
            inside = np.where(D[i] < radii[i])[0]
            for j in inside:
                diff = X_maj[j] - X_min[i]
                d = D[i, j]
                if d < tiny:                           # coincident point: random direction
                    diff = rng.choice([-1.0, 1.0], X_maj.shape[1]) * 1e-6
                    d = np.abs(diff).sum() if p == 1 else np.linalg.norm(diff)
                T[j] += (radii[i] - d) / d * diff
        moved = np.any(T != 0, axis=1)
        if self.translate:
            X_maj = X_maj + T

        # ---- Step 2: allocation (Eq. 1)
        raw, num = _allocate(radii, n_generate)
        if self.exact:
            rest = int(n_generate - num.sum())
            if rest > 0:
                num[np.argsort(-(raw - num))[:rest]] += 1

        # ---- Step 2: repartition with the class potential (Eqs. 2-3)
        D_mm = _cdist(X_min, X_min, p)
        syn, n_short = [], 0
        for i in range(n_min):
            if num[i] <= 0:
                continue
            local = X_min[D_mm[i] <= radii[i]]          # min^i (contains x_i itself)
            phi0 = _potential(X_min[i], local, self.eps, p)[0]
            C = _uniform_in_ball(X_min[i], radii[i], self.n_candidates, p, rng)
            if self.repartition == "paper":
                U = _potential(C, local, self.eps, p)
                R = phi0 + (U.max() - phi0) / self.n_partitions      # Eq. 3
                keep = C[U >= R] if self.region == "high" else C[U < R]
            else:
                # A-NRO: rank candidates by log-potential (no underflow) and keep
                # the top (or bottom) 1/n fraction; the seed is not the reference,
                # so the region can never be empty
                logU = _log_potential(C, local, self.eps, p)
                q = 1.0 / self.n_partitions
                if self.region == "high":
                    keep = C[logU >= np.quantile(logU, 1 - q)]
                else:
                    keep = C[logU <= np.quantile(logU, q)]
            if len(keep) >= num[i]:
                syn.append(keep[rng.choice(len(keep), num[i], replace=False)])
            else:
                n_short += 1
                if self.fill == "replace":
                    # CHOICE (same as public code): pool = retained candidates + seed,
                    # drawn with replacement so the requested count is met
                    pool = np.vstack([keep, X_min[i][None]]) if len(keep) else X_min[i][None]
                    syn.append(pool[rng.choice(len(pool), num[i], replace=True)])
                elif len(keep):                        # 'strict': keep what exists
                    syn.append(keep)
        X_syn = np.vstack(syn) if syn else np.empty((0, X_min.shape[1]))
        info = dict(radii=radii, num=num, n_moved=int(moved.sum()),
                    n_short=n_short)
        return X_maj, moved, X_syn, info


# ----------------------------------------------------------------------------
# Algorithm 2: MC-NRO with OVS decomposition + final cleaning
# ----------------------------------------------------------------------------
class MCNRO:
    """Multiclass Neighbourhood Repartition-based Oversampling.

    Parameters (paper names in brackets)
    ----------
    k1 : neighbours that define the neighbourhood radius [k1]
    k2 : neighbours used in the final overlap cleaning [k2]
    eps : spread of the Gaussian RBF [epsilon]
    n_partitions : number of target regions in Eq. 3 [n]
    n_candidates : candidate samples per seed [iterations]
    region : 'high' (near the seed, Fig. 3a) or 'low' (near moved majority, Fig. 3b)
    p : Lp norm for distances (paper: 1)
    cleaning : 'translate_remove' (paper default), 'translate', or 'none'
    scale : min-max scale internally (as in the authors' experiments) and map
            results back to the original units
    ordinal_cols, onehot_groups : OPTIONAL extension (not MC-NRO), see module doc
    """

    def __init__(self, k1=5, k2=5, eps=0.5, n_partitions=3, n_candidates=1000,
                 region="high", p=1, cleaning="translate_remove", scale=True,
                 exact=True, fill="replace", ordinal_cols=None,
                 onehot_groups=None, repartition="paper", clean_rule="any",
                 project_moved=False, random_state=None):
        assert cleaning in ("translate_remove", "translate", "none")
        assert clean_rule in ("any", "majority")
        self.repartition, self.clean_rule = repartition, clean_rule
        self.project_moved = project_moved
        self.k1, self.k2, self.eps = k1, k2, eps
        self.n_partitions, self.n_candidates = n_partitions, n_candidates
        self.region, self.p, self.cleaning, self.scale = region, p, cleaning, scale
        self.exact, self.fill = exact, fill
        self.ordinal_cols, self.onehot_groups = ordinal_cols, onehot_groups
        self.random_state = random_state

    # sklearn / imbalanced-learn style
    def fit_resample(self, X, y):
        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        lo, hi = X.min(0), X.max(0)
        span = np.where(hi - lo > 0, hi - lo, 1.0)
        Z = (X - lo) / span if self.scale else X.copy()

        classes, counts = np.unique(y, return_counts=True)
        order = classes[np.argsort(-counts, kind="stable")]      # descending size
        n_max = counts.max()

        # pool of all observations; each row keeps label / synthetic / moved flags
        P = Z.copy()
        L = y.copy()
        syn_flag = np.zeros(len(P), bool)
        moved_flag = np.zeros(len(P), bool)
        self.log_ = []

        bnro = BNRO(k1=self.k1, eps=self.eps, n_partitions=self.n_partitions,
                    n_candidates=self.n_candidates, region=self.region, p=self.p,
                    translate=self.cleaning != "none", exact=self.exact,
                    fill=self.fill, repartition=self.repartition, rng=rng)

        for i in range(1, len(order)):
            c = order[i]
            min_idx = np.where(L == c)[0]
            n_gen = int(n_max - len(min_idx))
            if n_gen <= 0:
                continue
            # combined majority: floor(|C1| / n_c) random rows from each larger class
            # (Alg. 2 line 12), drawn from the CURRENT pool incl. earlier synthetics
            per = int(np.floor(n_max / i))
            maj_idx = []
            for j in range(i):
                cand = np.where(L == order[j])[0]
                maj_idx.append(rng.choice(cand, min(per, len(cand)), replace=False))
            maj_idx = np.concatenate(maj_idx)

            X_maj_new, moved, X_syn, info = bnro.run(P[maj_idx], P[min_idx], n_gen)
            P[maj_idx] = X_maj_new                     # moved points re-enter the pool
            moved_flag[maj_idx[moved]] = True
            if len(X_syn):
                P = np.vstack([P, X_syn])
                L = np.concatenate([L, np.full(len(X_syn), c, dtype=L.dtype)])
                syn_flag = np.concatenate([syn_flag, np.ones(len(X_syn), bool)])
                moved_flag = np.concatenate([moved_flag, np.zeros(len(X_syn), bool)])
            self.log_.append(dict(cls=c, n_min=len(min_idx), n_maj=len(maj_idx),
                                  n_generated=len(X_syn), **{k: v for k, v in info.items()
                                                            if k not in ("radii", "num")}))

        # ---- final cleaning (Alg. 2 lines 17-22): a moved majority point is removed
        #      if its k2 nearest neighbours in the whole sample space are not all
        #      of its own class
        removed = np.zeros(len(P), bool)
        if self.cleaning == "translate_remove" and moved_flag.any():
            mv = np.where(moved_flag)[0]
            D = _cdist(P[mv], P, self.p)
            D[np.arange(len(mv)), mv] = np.inf         # exclude the point itself
            k2 = int(min(self.k2, len(P) - 1))
            nn = np.argsort(D, axis=1, kind="stable")[:, :k2]
            disagree = (L[nn] != L[mv][:, None])
            if self.clean_rule == "any":                   # paper: any foreign neighbour
                bad = disagree.any(axis=1)
            else:                                          # A-NRO: majority vote (ENN-like)
                bad = disagree.sum(axis=1) > k2 / 2.0
            removed[mv[bad]] = True

        keep = ~removed
        P, L, syn_flag = P[keep], L[keep], syn_flag[keep]
        moved_kept = moved_flag[keep]
        Xo = P * span + lo if self.scale else P

        # ---- OPTIONAL extension: project synthetic rows to valid questionnaire values
        if self.ordinal_cols is not None or self.onehot_groups is not None:
            sel = syn_flag | (moved_kept if self.project_moved else False)
            S = Xo[sel]
            for col in (self.ordinal_cols or []):
                S[:, col] = np.clip(np.rint(S[:, col]), lo[col], hi[col])
            for grp in (self.onehot_groups or []):
                g = np.asarray(grp)
                hot = np.argmax(S[:, g], axis=1)
                S[:, g] = 0.0
                S[np.arange(len(S)), g[hot]] = 1.0
            Xo[sel] = S

        self.n_removed_ = int(removed.sum())
        self.n_moved_ = int(moved_flag.sum())
        self.synthetic_mask_ = syn_flag
        return Xo, L


class ANRO(MCNRO):
    """A-NRO: adaptive variant of MC-NRO proposed for small ordinal survey data
    (authors' own modification, NOT part of Shen et al., 2024).

    Differences from MCNRO (each can be switched off for ablation):
      repartition='quantile' : rank candidates by log-potential and keep the
                               top 1/n fraction, instead of Eq. 3 (which is
                               anchored at the seed and is empty whenever the
                               seed has the highest potential)
      clean_rule='majority'  : remove a moved point only if most of its k2
                               neighbours belong to other classes (paper: any)
      ordinal projection     : synthetic AND moved rows are mapped back to valid
                               questionnaire values (pass ordinal_cols/onehot_groups)
    """

    def __init__(self, repartition="quantile", clean_rule="majority",
                 project_moved=True, **kw):
        super().__init__(repartition=repartition, clean_rule=clean_rule,
                         project_moved=project_moved, **kw)

In [ ]:
%%writefile samplers.py
"""Reference oversamplers (multiclass, one-vs-rest, oversample each class to n_max).

Written from the original papers because imbalanced-learn is not installed in
this environment. For the final manuscript, re-run with imbalanced-learn
(RandomOverSampler, SMOTE, BorderlineSMOTE(kind='borderline-1'), ADASYN) and
report its version.

  ROS                : random duplication
  SMOTE              : Chawla et al. (2002), k = 5
  Borderline-SMOTE-1 : Han et al. (2005), m = 10, k = 5
  ADASYN             : He et al. (2008), K = 5, beta = 1
"""
import numpy as np
from scipy.spatial.distance import cdist


def _interp(Z, seeds, k, n_per_seed, rng):
    """SMOTE interpolation from seeds (indices into Z) toward minority NNs."""
    if len(Z) < 2:
        return np.repeat(Z[seeds], n_per_seed, axis=0)
    kk = min(k, len(Z) - 1)
    D = cdist(Z, Z)
    np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, axis=1, kind="stable")[:, :kk]
    a = np.repeat(seeds, n_per_seed)
    b = nn[a, rng.integers(0, kk, len(a))]
    g = rng.random((len(a), 1))
    return Z[a] + g * (Z[b] - Z[a])


def _alloc_even(n_seeds, n, rng):
    base = np.full(n_seeds, n // n_seeds)
    base[rng.choice(n_seeds, n % n_seeds, replace=False)] += 1
    return base


def resample(X, y, method, k=5, m=10, random_state=None):
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, float); y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    n_max = counts.max()
    Xs, ys, info = [X], [y], {}
    for c, nc in zip(classes, counts):
        n = n_max - nc
        if n <= 0:
            continue
        idx = np.where(y == c)[0]
        Z = X[idx]
        if method == "ros":
            new = Z[rng.choice(len(Z), n)]
        elif method == "smote":
            per = _alloc_even(len(Z), n, rng)
            new = np.vstack([_interp(Z, np.array([i]), k, per[i], rng) for i in range(len(Z)) if per[i]])
        elif method in ("borderline", "adasyn"):
            K = m if method == "borderline" else k
            Dall = cdist(Z, X)
            Dall[np.arange(len(Z)), idx] = np.inf
            nnall = np.argsort(Dall, axis=1, kind="stable")[:, :min(K, len(X) - 1)]
            n_foreign = (y[nnall] != c).sum(1)
            if method == "borderline":
                danger = np.where((n_foreign >= K / 2) & (n_foreign < K))[0]
                info[c] = dict(n_danger=len(danger))
                if len(danger) == 0:           # same behaviour as imbalanced-learn: nothing to do
                    continue
                per = _alloc_even(len(danger), n, rng)
                new = np.vstack([_interp(Z, np.array([danger[i]]), k, per[i], rng)
                                 for i in range(len(danger)) if per[i]])
            else:
                r = n_foreign / K
                info[c] = dict(n_hard=int((r > 0).sum()))
                if r.sum() == 0:
                    continue
                g = np.rint(r / r.sum() * n).astype(int)
                new = np.vstack([_interp(Z, np.array([i]), k, g[i], rng) for i in range(len(Z)) if g[i]])
        else:
            raise ValueError(method)
        Xs.append(new); ys.append(np.full(len(new), c, dtype=y.dtype))
    return np.vstack(Xs), np.concatenate(ys)

In [ ]:
%%writefile anro_exp.py
"""Experiment core for 'Beyond the Baseline' (paper 1).

Protocol = IKADO baseline paper:
  19 non-label ordinal items + 7 respondent/business characteristics (26 predictors);
  imputation + one-hot fitted on the training fold; repeated stratified 5-fold x 10
  (random_state=42); RF 500 trees, gini, sqrt, bootstrap, random_state=42.
  Every resampler is applied to the TRAINING fold only.
The data path is read from the environment variable ANRO_DATA so that joblib
worker processes can load the same data.
"""
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import f1_score

from encode import load, NONLABEL, DEMO
from mcnro import MCNRO, ANRO

try:
    from imblearn.over_sampling import RandomOverSampler, SMOTE, SMOTENC, BorderlineSMOTE, ADASYN
    HAVE_IMBLEARN = True
except ImportError:                          # local fallback (not for the manuscript)
    HAVE_IMBLEARN = False
    from samplers import resample as _own_resample

_DEFAULT = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "data", "msme_digital_readiness_anonymized.csv")
X, demo, drs, y = load(os.environ.get("ANRO_DATA", _DEFAULT))
NL, PRED = NONLABEL, NONLABEL + DEMO
D = pd.concat([X[NL], demo], axis=1)
Y = y.values
CV = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42).split(D, Y))
RF = dict(n_estimators=500, criterion="gini", max_features="sqrt", bootstrap=True, random_state=42)
NRO_PAR = dict(k1=5, k2=5, eps=0.5, n_partitions=3, n_candidates=1000)
PERM_REPEATS = int(os.environ.get("ANRO_PERM_REPEATS", 10))

#            name: (sampler, extra RF kwargs, decision rule, tune, OOF permutation)
METHODS = {
    "baseline":        (None, {}, "argmax", False, True),
    "rf_tuned":        (None, {}, "argmax", True, False),
    "class_weight":    (None, {"class_weight": "balanced"}, "argmax", False, True),
    "threshold_prior": (None, {}, "prior", False, True),
    "ros":             ("ros", {}, "argmax", False, False),
    "smote":           ("smote", {}, "argmax", False, True),
    "smotenc":         ("smotenc", {}, "argmax", False, True),
    "borderline":      ("borderline", {}, "argmax", False, False),
    "adasyn":          ("adasyn", {}, "argmax", False, False),
    "mcnro":           ("mcnro", {}, "argmax", False, True),
    "mcnro+quantile":  ("mcnro+quantile", {}, "argmax", False, False),
    "mcnro+majclean":  ("mcnro+majclean", {}, "argmax", False, False),
    "mcnro+ordinal":   ("mcnro+ordinal", {}, "argmax", False, False),
    "anro-noproj":     ("anro-noproj", {}, "argmax", False, False),
    "anro":            ("anro", {}, "argmax", False, True),
}
LABEL = {"baseline": "RF baseline (IKADO config)", "rf_tuned": "RF tuned (nested CV)",
         "class_weight": "RF class_weight=balanced", "threshold_prior": "RF + prior-corrected threshold",
         "ros": "ROS", "smote": "SMOTE", "smotenc": "SMOTE-NC", "borderline": "Borderline-SMOTE-1",
         "adasyn": "ADASYN", "mcnro": "MC-NRO (paper-faithful)",
         "mcnro+quantile": "MC-NRO + quantile repartition",
         "mcnro+majclean": "MC-NRO + majority-vote cleaning",
         "mcnro+ordinal": "MC-NRO + ordinal projection",
         "anro-noproj": "A-NRO without ordinal projection", "anro": "A-NRO (full)"}


# ---------------------------------------------------------------- preprocessing
class StdPrep:
    """Median / mode imputation + one-hot (same as the IKADO pipeline)."""
    def fit(self, Dtr):
        self.ct = ColumnTransformer([
            ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), NL),
            ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), DEMO)],
            remainder="drop").fit(Dtr)
        cats = self.ct.named_transformers_["categorical"].named_steps["onehot"].categories_
        self.owner, self.groups, start = list(range(len(NL))), [], len(NL)
        for j, cs in enumerate(cats):
            self.groups.append(list(range(start, start + len(cs))))
            self.owner += [len(NL) + j] * len(cs)
            start += len(cs)
        self.owner = np.array(self.owner)
        return self

    def transform(self, Dx):
        M = self.ct.transform(Dx)
        return M.toarray() if hasattr(M, "toarray") else np.asarray(M, float)


class NCPrep:
    """For SMOTE-NC: impute, ordinal-code categories, oversample, then one-hot."""
    def fit(self, Dtr):
        self.num_imp = SimpleImputer(strategy="median").fit(Dtr[NL])
        self.cat_imp = SimpleImputer(strategy="most_frequent").fit(Dtr[DEMO])
        self.ord = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(
            self.cat_imp.transform(Dtr[DEMO]))
        n_levels = [len(c) for c in self.ord.categories_]
        self.oh = OneHotEncoder(categories=[list(range(n)) for n in n_levels],
                                handle_unknown="ignore").fit(np.zeros((1, len(DEMO))))
        self.owner, self.groups, start = list(range(len(NL))), [], len(NL)
        for j, n in enumerate(n_levels):
            self.groups.append(list(range(start, start + n)))
            self.owner += [len(NL) + j] * n
            start += n
        self.owner = np.array(self.owner)
        return self

    def raw(self, Dx):            # numeric + ordinal codes (for SMOTE-NC)
        return np.hstack([self.num_imp.transform(Dx[NL]),
                          self.ord.transform(self.cat_imp.transform(Dx[DEMO]))])

    def encode(self, R):          # raw -> model matrix
        oh = self.oh.transform(np.rint(R[:, len(NL):]).astype(int))
        oh = oh.toarray() if hasattr(oh, "toarray") else oh
        return np.hstack([R[:, :len(NL)], oh])

    def transform(self, Dx):
        return self.encode(self.raw(Dx))


# ---------------------------------------------------------------- samplers
def _imb(method, Xt, yt, seed):
    kmin = int(np.bincount(yt).min())
    k = max(1, min(5, kmin - 1))
    if not HAVE_IMBLEARN:
        return _own_resample(Xt, yt, method, random_state=seed)
    s = {"ros": lambda: RandomOverSampler(random_state=seed),
         "smote": lambda: SMOTE(k_neighbors=k, random_state=seed),
         "borderline": lambda: BorderlineSMOTE(kind="borderline-1", k_neighbors=k,
                                               m_neighbors=10, random_state=seed),
         "adasyn": lambda: ADASYN(n_neighbors=k, random_state=seed)}[method]()
    return s.fit_resample(Xt, yt)


def resample_fold(method, Xt, yt, groups, seed, prep=None, Dtr=None):
    diag = {}
    if method in ("ros", "smote", "borderline", "adasyn"):
        try:
            Xr, yr = _imb(method, Xt, yt, seed)
        except (RuntimeError, ValueError) as e:          # e.g. ADASYN with no hard samples
            Xr, yr = Xt, yt
            diag["sampler_failed"] = str(e)[:120]
        diag["train_counts"] = np.bincount(yr, minlength=3).tolist()
        return Xr, yr, diag
    if method == "smotenc":
        R = prep.raw(Dtr)
        k = max(1, min(5, int(np.bincount(yt).min()) - 1))
        cat_idx = list(range(len(NL), len(NL) + len(DEMO)))
        if HAVE_IMBLEARN:
            Rr, yr = SMOTENC(categorical_features=cat_idx, k_neighbors=k,
                             random_state=seed).fit_resample(R, yt)
        else:
            Rr, yr = R, yt
            diag["sampler_failed"] = "imbalanced-learn not installed"
        diag["train_counts"] = np.bincount(yr, minlength=3).tolist()
        return prep.encode(Rr), yr, diag
    ordc = list(range(len(NL)))
    s = {
        "mcnro":          lambda: MCNRO(**NRO_PAR, random_state=seed),
        "mcnro+quantile": lambda: MCNRO(**NRO_PAR, repartition="quantile", random_state=seed),
        "mcnro+majclean": lambda: MCNRO(**NRO_PAR, clean_rule="majority", random_state=seed),
        "mcnro+ordinal":  lambda: MCNRO(**NRO_PAR, ordinal_cols=ordc, onehot_groups=groups,
                                        project_moved=True, random_state=seed),
        "anro-noproj":    lambda: ANRO(**NRO_PAR, random_state=seed),
        "anro":           lambda: ANRO(**NRO_PAR, ordinal_cols=ordc, onehot_groups=groups,
                                       random_state=seed),
    }[method]()
    Xr, yr = s.fit_resample(Xt, yt)
    syn = Xr[s.synthetic_mask_]
    diag = dict(n_moved=s.n_moved_, n_removed=s.n_removed_,
                n_short=[d["n_short"] for d in s.log_],
                uniq_syn_high=len(np.unique(np.round(syn[yr[s.synthetic_mask_] == 2], 8), axis=0)),
                n_syn_high=int((yr[s.synthetic_mask_] == 2).sum()),
                train_counts=np.bincount(yr, minlength=3).tolist())
    return Xr, yr, diag


# ---------------------------------------------------------------- model
def tune_rf(Xt, yt, seed):
    grid = [dict(max_features=mf, min_samples_leaf=ml, max_depth=md)
            for mf in ("sqrt", 0.5) for ml in (1, 3, 5) for md in (None, 8)]
    inner = StratifiedKFold(3, shuffle=True, random_state=seed)
    best, best_s = None, -1.0
    for g in grid:
        s = []
        for a, b in inner.split(Xt, yt):
            m = RandomForestClassifier(n_estimators=200, criterion="gini", bootstrap=True,
                                       random_state=42, n_jobs=1, **g).fit(Xt[a], yt[a])
            s.append(f1_score(yt[b], m.predict(Xt[b]), average="macro"))
        if np.mean(s) > best_s:
            best, best_s = g, float(np.mean(s))
    return best


def decide(proba, rule, prior):
    return (proba / prior).argmax(1) if rule == "prior" else proba.argmax(1)


def fold_job(fi, name, perm=None):
    samp, rfkw, rule, tune, do_perm = METHODS[name]
    do_perm = do_perm if perm is None else perm
    tr, te = CV[fi]
    prep = (NCPrep() if samp == "smotenc" else StdPrep()).fit(D.iloc[tr])
    Xt, Xs = prep.transform(D.iloc[tr]), prep.transform(D.iloc[te])
    yt = Y[tr]
    prior = np.bincount(yt, minlength=3) / len(yt)
    diag = {}
    if samp is not None:
        Xt, yt, diag = resample_fold(samp, Xt, yt, prep.groups, seed=1000 + fi,
                                     prep=prep, Dtr=D.iloc[tr])
    kw = dict(RF); kw.update(rfkw)
    if tune:
        best = tune_rf(Xt, yt, seed=fi); kw.update(best); diag["best"] = best
    m = RandomForestClassifier(n_jobs=1, **kw).fit(Xt, yt)
    proba = m.predict_proba(Xs)
    pred = decide(proba, rule, prior)
    gini26 = np.bincount(prep.owner, weights=m.feature_importances_, minlength=len(PRED))
    perm26 = None
    if do_perm:
        rng = np.random.default_rng(fi)
        base = f1_score(Y[te], pred, average="macro")
        perm26 = np.zeros(len(PRED))
        for j, col in enumerate(PRED):
            drops = []
            for _ in range(PERM_REPEATS):
                Dp = D.iloc[te].copy()
                Dp[col] = rng.permutation(Dp[col].values)
                pp = decide(m.predict_proba(prep.transform(Dp)), rule, prior)
                drops.append(base - f1_score(Y[te], pp, average="macro"))
            perm26[j] = np.mean(drops)
    return dict(fold=fi, rep=fi // 5, te=te, pred=pred, proba=proba,
                gini26=gini26, perm26=perm26, diag=diag)


def sens_job(fi, algo, k1, eps, n):
    tr, te = CV[fi]
    prep = StdPrep().fit(D.iloc[tr])
    Xt, Xs = prep.transform(D.iloc[tr]), prep.transform(D.iloc[te])
    par = dict(k1=k1, k2=5, eps=eps, n_partitions=n, n_candidates=1000, random_state=1000 + fi)
    s = MCNRO(**par) if algo == "MC-NRO" else ANRO(**par, ordinal_cols=list(range(len(NL))),
                                                    onehot_groups=prep.groups)
    Xr, yr = s.fit_resample(Xt, Y[tr])
    p = RandomForestClassifier(n_jobs=1, **RF).fit(Xr, yr).predict(Xs)
    return Y[te], p


In [ ]:
%%writefile anro_analysis.py
"""Tables for the manuscript, computed from results/<method>.pkl."""
import os, pickle, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score,
                             matthews_corrcoef, precision_recall_fscore_support, roc_auc_score,
                             mean_absolute_error, confusion_matrix)
from anro_exp import METHODS, LABEL, PRED, Y

IKADO_SET = {"Q13", "Q15", "Q16", "Q7"}
IKADO_PUBLISHED = dict(Accuracy=0.7384, BalAcc=0.4555, MacroF1=0.4470, Kappa=0.3739, MCC=0.3825)


def load(name, folder="results"):
    return pickle.load(open(f"{folder}/{name}.pkl", "rb"))


def stack(res):
    t = np.concatenate([Y[r["te"]] for r in res])
    p = np.concatenate([r["pred"] for r in res])
    pr = np.vstack([r["proba"] for r in res])
    idx = np.concatenate([r["te"] for r in res])
    return t, p, pr, idx


def brier(t, pr):
    onehot = np.eye(pr.shape[1])[t]
    return float(np.mean(np.sum((pr - onehot) ** 2, axis=1)))


def pooled(res):
    t, p, pr, _ = stack(res)
    P, R, F, _ = precision_recall_fscore_support(t, p, labels=[0, 1, 2], zero_division=0)
    return dict(Accuracy=accuracy_score(t, p), BalAcc=balanced_accuracy_score(t, p),
                MacroF1=f1_score(t, p, average="macro"), WeightedF1=f1_score(t, p, average="weighted"),
                Kappa=cohen_kappa_score(t, p), MCC=matthews_corrcoef(t, p),
                QWK=cohen_kappa_score(t, p, weights="quadratic"), MAE=mean_absolute_error(t, p),
                MacroAUC=roc_auc_score(t, pr, multi_class="ovr", average="macro"),
                AUC_High=roc_auc_score((t == 2).astype(int), pr[:, 2]), Brier=brier(t, pr),
                P_Low=P[0], R_Low=R[0], F1_Low=F[0], P_Med=P[1], R_Med=R[1], F1_Med=F[1],
                P_High=P[2], R_High=R[2], F1_High=F[2],
                Extreme_errors=int(np.sum(np.abs(t - p) == 2)))


def per_fold(res, metric):
    f = (lambda t, p: f1_score(t, p, average="macro")) if metric == "f1" else balanced_accuracy_score
    return np.array([f(Y[r["te"]], r["pred"]) for r in res])


def per_rep(res):
    rows = []
    for k in range(10):
        rr = [r for r in res if r["rep"] == k]
        t = np.concatenate([Y[r["te"]] for r in rr]); p = np.concatenate([r["pred"] for r in rr])
        rows.append([f1_score(t, p, average="macro"), balanced_accuracy_score(t, p), np.mean(p[t == 2] == 2)])
    return np.array(rows)


def corrected_t(d, test_frac=0.2):
    """Nadeau & Bengio (2003) corrected resampled t-test on per-fold differences."""
    n, m, v = len(d), d.mean(), d.var(ddof=1)
    if v == 0:
        return 0.0, 1.0
    t = m / np.sqrt((1 / n + test_frac / (1 - test_frac)) * v)
    return float(t), float(2 * stats.t.sf(abs(t), n - 1))


def respondent_votes(res):
    votes = np.zeros((len(Y), 3), int)
    for r in res:
        votes[r["te"], r["pred"]] += 1
    return votes


def respondent(res):
    v = respondent_votes(res); maj = v.argmax(1)
    out = {f"{n} correct (majority vote)": f"{int(np.sum((maj == c) & (Y == c)))}/{int(np.sum(Y == c))}"
           for c, n in [(0, "Low"), (1, "Medium"), (2, "High")]}
    out["High hit at least once (of 10 reps)"] = f"{int(np.sum(v[Y == 2, 2] > 0))}/9"
    return out


def bootstrap(res, base_res=None, B=2000, seed=0):
    """Respondent-level stratified bootstrap of pooled OOF predictions (all 10 reps per respondent)."""
    rng = np.random.default_rng(seed)
    t, p, _, idx = stack(res)
    if base_res is not None:
        tb, pb, _, idxb = stack(base_res)
        order_b = np.argsort(idxb, kind="stable")
    order = np.argsort(idx, kind="stable")
    t, p = t[order].reshape(len(Y), -1), p[order].reshape(len(Y), -1)     # respondent x reps
    if base_res is not None:
        pb = pb[order_b].reshape(len(Y), -1)
    cls_idx = [np.where(Y == c)[0] for c in range(3)]
    f1s, rh, d = [], [], []
    for _ in range(B):
        s = np.concatenate([rng.choice(ix, len(ix)) for ix in cls_idx])
        tt, pp = t[s].ravel(), p[s].ravel()
        f1s.append(f1_score(tt, pp, average="macro")); rh.append(np.mean(pp[tt == 2] == 2))
        if base_res is not None:
            d.append(f1s[-1] - f1_score(tt, pb[s].ravel(), average="macro"))
    q = lambda a: f"[{np.percentile(a, 2.5):.3f}, {np.percentile(a, 97.5):.3f}]"
    out = {"MacroF1 95% CI": q(f1s), "Recall High 95% CI": q(rh)}
    if base_res is not None:
        out["dMacroF1 vs baseline 95% CI"] = q(d)
        out["P(dMacroF1 > 0)"] = float(np.mean(np.array(d) > 0))
    return out


def stability(res, key):
    arr = [r[key] for r in res if r[key] is not None]
    if not arr:
        return None
    imp = np.vstack(arr)
    ranks = (-imp).argsort(1).argsort(1) + 1
    mean_imp = imp.mean(0)
    top4 = [PRED[i] for i in np.argsort(-mean_imp)[:4]]
    return dict(top4=top4, jaccard=len(set(top4) & IKADO_SET) / len(set(top4) | IKADO_SET),
                top10=({q: float(np.mean(ranks[:, PRED.index(q)] <= 10)) for q in ["Q13", "Q15", "Q16", "Q7", "Q2"]}),
                mean_imp=mean_imp)


def build_tables(folder="results", B=2000):
    avail = [n for n in METHODS if os.path.exists(f"{folder}/{n}.pkl")]
    R = {n: load(n, folder) for n in avail}
    base = R["baseline"]
    bf1, bba = per_fold(base, "f1"), per_fold(base, "ba")
    thr = R.get("threshold_prior")
    tables = {k: [] for k in ["performance", "per_repetition", "significance", "bootstrap_CI",
                              "respondent_level", "confusion_matrices", "nro_diagnostics",
                              "indicator_stability", "sampler_notes"]}
    for n in avail:
        res, L = R[n], LABEL[n]
        tables["performance"].append({"Method": L, **pooled(res)})
        pr = per_rep(res)
        tables["per_repetition"].append({"Method": L,
            "MacroF1 mean±sd": f"{pr[:,0].mean():.3f} ± {pr[:,0].std(ddof=1):.3f}",
            "BalAcc mean±sd": f"{pr[:,1].mean():.3f} ± {pr[:,1].std(ddof=1):.3f}",
            "Recall High mean±sd": f"{pr[:,2].mean():.3f} ± {pr[:,2].std(ddof=1):.3f}"})
        f1, ba = per_fold(res, "f1"), per_fold(res, "ba")
        row = {"Method": L, "dMacroF1 vs baseline": (f1 - bf1).mean(), "p corrected t (F1)": corrected_t(f1 - bf1)[1],
               "dBalAcc vs baseline": (ba - bba).mean(), "p corrected t (BA)": corrected_t(ba - bba)[1]}
        if thr is not None:
            tf1 = per_fold(thr, "f1")
            row.update({"dMacroF1 vs threshold": (f1 - tf1).mean(), "p vs threshold": corrected_t(f1 - tf1)[1]})
        tables["significance"].append(row)
        tables["bootstrap_CI"].append({"Method": L, **bootstrap(res, None if n == "baseline" else base, B=B)})
        tables["respondent_level"].append({"Method": L, **respondent(res)})
        t, p, _, _ = stack(res); cm = confusion_matrix(t, p, labels=[0, 1, 2])
        tables["confusion_matrices"].append({"Method": L, **{f"{a}->{b}": int(cm[i, j])
            for i, a in enumerate(["Low", "Med", "High"]) for j, b in enumerate(["Low", "Med", "High"])}})
        ds = [r["diag"] for r in res if r["diag"]]
        if ds and "n_moved" in ds[0]:
            tc = np.array([d["train_counts"] for d in ds])
            tables["nro_diagnostics"].append({"Method": L,
                "moved/fold": np.mean([d["n_moved"] for d in ds]),
                "removed/fold": np.mean([d["n_removed"] for d in ds]),
                "short seeds Medium": np.mean([d["n_short"][0] for d in ds]),
                "short seeds High": np.mean([d["n_short"][1] for d in ds]),
                "unique synthetic High / generated": f"{np.mean([d['uniq_syn_high'] for d in ds]):.1f} / {np.mean([d['n_syn_high'] for d in ds]):.1f}",
                "train Low/Med/High after": "/".join(f"{v:.1f}" for v in tc.mean(0))})
        fails = [d.get("sampler_failed") for d in ds if d.get("sampler_failed")]
        if fails or n == "rf_tuned":
            note = f"{len(fails)}/50 folds sampler failed (no resampling): {fails[0]}" if fails else ""
            if n == "rf_tuned":
                note = "best params: " + "; ".join(f"{k} x{v}" for k, v in
                                                     pd.Series([str(d.get('best')) for d in ds]).value_counts().items())
            tables["sampler_notes"].append({"Method": L, "Note": note})
        for key, lab in [("gini26", "Gini"), ("perm26", "OOF permutation")]:
            s = stability(res, key)
            if s:
                tables["indicator_stability"].append({"Method": L, "Importance": lab,
                    "Top-4": ", ".join(s["top4"]), "Jaccard top-4 vs IKADO set": s["jaccard"],
                    **{f"Top-10 freq {q}": v for q, v in s["top10"].items()}})
    tables = {k: pd.DataFrame(v) for k, v in tables.items()}
    b = tables["performance"].iloc[0]
    tables["baseline_check"] = pd.DataFrame([{"Metric": k, "IKADO published": v, "This run": b[k],
                                              "abs diff": abs(b[k] - v)} for k, v in IKADO_PUBLISHED.items()])
    return tables

In [ ]:
%%writefile test_mcnro.py
"""Property tests for mcnro.py. Run: python test_mcnro.py"""
import numpy as np
from scipy.spatial.distance import cdist
from mcnro import MCNRO, BNRO, _uniform_in_ball, _potential

rng = np.random.default_rng(1)
ok = lambda name, cond: print(("PASS " if cond else "FAIL ") + name) or cond
res = []

# synthetic 3-class problem, 6 features
X = np.vstack([rng.normal(0, 1, (120, 6)), rng.normal(1.5, 1, (30, 6)), rng.normal(3, 1, (8, 6))])
y = np.array([0] * 120 + [1] * 30 + [2] * 8)

# 1 uniform-in-ball: fraction inside r/2 must be ~0.5**m
for p in (1, 2):
    pts = _uniform_in_ball(np.zeros(6), 1.0, 200000, p, rng)
    norm = np.abs(pts).sum(1) if p == 1 else np.linalg.norm(pts, axis=1)
    res.append(ok(f"uniform L{p} ball: inside<=r and P(<r/2)~{0.5**6:.4f} (got {(norm<0.5).mean():.4f})",
                  norm.max() <= 1 + 1e-9 and abs((norm < 0.5).mean() - 0.5 ** 6) < 0.003))

# 2 balance + only moved majority rows removed + minority never removed
m = MCNRO(k1=5, k2=5, eps=0.5, random_state=0)
Xr, yr = m.fit_resample(X, y)
cnt = np.bincount(yr)
res.append(ok(f"minority classes balanced to n_max: {cnt.tolist()}", cnt[1] == 120 and cnt[2] == 120))
res.append(ok(f"removed rows ({m.n_removed_}) <= moved rows ({m.n_moved_})", m.n_removed_ <= m.n_moved_))
res.append(ok("no NaN/inf", np.isfinite(Xr).all()))

# 3 every synthetic point lies inside some seed's L1 neighbourhood (scaled space)
lo, hi = X.min(0), X.max(0); Z = (X - lo) / (hi - lo)
b = BNRO(k1=5, eps=0.5, rng=np.random.default_rng(0))
Xm, mv, S, info = b.run(Z[y != 2], Z[y == 2], 112)
d = cdist(S, Z[y == 2], "cityblock")
res.append(ok("synthetic within radius of a seed", np.all((d <= info["radii"][None] + 1e-9).any(1))))
res.append(ok(f"generated exactly n ({len(S)})", len(S) == 112))

# 4 moved majority points end on/outside every neighbourhood that contained them
D_before = cdist(Z[y == 2], Z[y != 2], "cityblock")
D_after = cdist(Z[y == 2], Xm, "cityblock")
inside = D_before < info["radii"][:, None]
single = np.where(inside.sum(0) == 1)[0]            # majority rows inside exactly one ball
owner = inside[:, single].argmax(0)
res.append(ok(f"first cleaning moves majority to the ball boundary ({len(single)} rows checked)",
              len(single) > 0 and np.allclose(D_after[owner, single], info["radii"][owner], atol=1e-8)))

# 5 high-region rule: retained candidates satisfy U >= R (Eq. 3)
xi = Z[y == 2][0]; local = Z[y == 2][cdist(xi[None], Z[y == 2], "cityblock")[0] <= info["radii"][0]]
C = _uniform_in_ball(xi, info["radii"][0], 2000, 1, rng)
U = _potential(C, local, 0.5, 1); phi0 = _potential(xi, local, 0.5, 1)[0]
R = phi0 + (U.max() - phi0) / 3
res.append(ok(f"Eq.3 threshold between phi0 and max(U): phi0={phi0:.3f} R={R:.3f} maxU={U.max():.3f}",
              min(phi0, U.max()) - 1e-12 <= R <= max(phi0, U.max()) + 1e-12))

# 6 reproducibility
a = MCNRO(random_state=7).fit_resample(X, y)[0]; c = MCNRO(random_state=7).fit_resample(X, y)[0]
res.append(ok("reproducible with random_state", np.array_equal(a, c)))

# 7 optional ordinal projection gives valid Likert values on synthetic rows
Xo = np.clip(np.rint(X + 3), 1, 5)
mo = MCNRO(random_state=0, ordinal_cols=list(range(6)))
Xp, yp = mo.fit_resample(Xo, y)
S = Xp[mo.synthetic_mask_]
res.append(ok("ordinal projection -> integers in [1,5]", np.all(S == np.rint(S)) and S.min() >= 1 and S.max() <= 5))

# 8 A-NRO: quantile repartition never empty -> no short seeds, synthetic rows distinct
from mcnro import ANRO
an = ANRO(random_state=0); Xa, ya = an.fit_resample(X, y)
shorts = [d["n_short"] for d in an.log_]
Sa = Xa[an.synthetic_mask_]
res.append(ok(f"A-NRO no short seeds {shorts}, unique synthetic {len(np.unique(Sa.round(10), axis=0))}/{len(Sa)}",
              sum(shorts) == 0 and len(np.unique(Sa.round(10), axis=0)) == len(Sa)))
gen = [d["n_generated"] for d in an.log_]
res.append(ok(f"A-NRO generates n_max - n_c per class before cleaning ({gen})", gen == [90, 112]))

# 9 majority-vote cleaning removes no more than the paper's 'any' rule (same seed)
m_any = MCNRO(random_state=3, repartition="quantile", clean_rule="any"); m_any.fit_resample(X, y)
m_maj = MCNRO(random_state=3, repartition="quantile", clean_rule="majority"); m_maj.fit_resample(X, y)
res.append(ok(f"majority rule removes <= any rule ({m_maj.n_removed_} <= {m_any.n_removed_})",
              m_maj.n_removed_ <= m_any.n_removed_))

# 10 A-NRO projection also covers moved rows: every row is on the Likert lattice
ap = ANRO(random_state=0, ordinal_cols=list(range(6))); Xq, yq = ap.fit_resample(Xo, y)
res.append(ok("A-NRO: all rows (synthetic + moved) integer in [1,5]",
              np.all(Xq == np.rint(Xq)) and Xq.min() >= 1 and Xq.max() <= 5))

print(f"\n{sum(res)}/{len(res)} passed")

In [ ]:
# SEL 10 — Uji properti MC-NRO / A-NRO (harus 15/15 PASS)
!python test_mcnro.py

In [ ]:
# SEL 11 — Validasi data (harus sama dengan IKADO: 108/42/9 dan rata-rata item label)
import numpy as np, importlib, anro_exp as E
importlib.reload(E)
counts = np.bincount(E.Y).tolist()
means = E.X[["Q8","Q9","Q10","Q12","Q14"]].mean().round(4).tolist()
print("kelas Low/Med/High:", counts, "| rata-rata Q8,Q9,Q10,Q12,Q14:", means)
assert counts == [108, 42, 9], "Distribusi kelas tidak sama dengan IKADO"
assert means == [3.2516, 1.8931, 1.6855, 2.1447, 1.7233], "Encoding item label berbeda dari IKADO"
assert E.HAVE_IMBLEARN, "imbalanced-learn tidak terpasang"
print("✓ data identik dengan IKADO; imbalanced-learn aktif")

In [ ]:
# SEL 12 — Jalankan baseline dulu dan cocokkan dengan angka terbit IKADO
import os, pickle, time
from joblib import Parallel, delayed
import anro_analysis as A

def run_method(name):
    path = f"{RESULTS_DIR}/{name}.pkl"
    if os.path.exists(path):
        print(f"skip {name} (sudah ada)"); return
    t0 = time.time()
    res = Parallel(n_jobs=-1)(delayed(E.fold_job)(fi, name) for fi in range(len(E.CV)))
    pickle.dump(res, open(path, "wb"))
    print(f"✓ {name:16s} {time.time()-t0:6.0f} s")

run_method("baseline")
chk = A.pooled(A.load("baseline", RESULTS_DIR))
print()
for k, v in A.IKADO_PUBLISHED.items():
    flag = "✓" if abs(chk[k] - v) < 1e-4 else "≠"
    print(f"{flag} {k:9s} terbit {v:.4f} | run ini {chk[k]:.4f}")
MATCH = all(abs(chk[k] - v) < 1e-4 for k, v in A.IKADO_PUBLISHED.items())
print("\nBASELINE IDENTIK DENGAN IKADO" if MATCH else
      "\nBaseline belum identik. Eksperimen tetap valid (semua metode memakai run yang sama), "
      "tetapi catat selisih ini dan kirimkan bersama hasil.")

In [ ]:
# SEL 13 — Jalankan semua metode lain (±45–75 menit). Aman dijalankan ulang bila terputus.
for name in E.METHODS:
    run_method(name)
print("✓ semua metode selesai")

In [ ]:
# SEL 14 — Ringkasan tabel
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
T = A.build_tables(RESULTS_DIR, B=2000)
for k in ["baseline_check", "performance", "bootstrap_CI", "significance", "respondent_level",
          "nro_diagnostics", "indicator_stability", "sampler_notes"]:
    print(f"\n=== {k} ===")
    display(T[k].round(4))

In [ ]:
# SEL 15 — Analisis sensitivitas parameter MC-NRO vs A-NRO (±15–25 menit)
RUN_SENSITIVITY = True
import itertools, numpy as np, pandas as pd, os
from sklearn.metrics import f1_score, balanced_accuracy_score
sens_path = f"{RESULTS_DIR}/sensitivity.csv"
if RUN_SENSITIVITY and not os.path.exists(sens_path):
    rows = []
    for algo, k1, eps, n in itertools.product(["MC-NRO", "A-NRO"], [3, 5, 7], [0.1, 0.5, 2.5], [2, 3, 5]):
        out = Parallel(n_jobs=-1)(delayed(E.sens_job)(fi, algo, k1, eps, n) for fi in range(50))
        t = np.concatenate([o[0] for o in out]); p = np.concatenate([o[1] for o in out])
        rows.append(dict(algo=algo, k1=k1, eps=eps, n=n, MacroF1=f1_score(t, p, average="macro"),
                         BalAcc=balanced_accuracy_score(t, p), R_Low=np.mean(p[t == 0] == 0),
                         R_Med=np.mean(p[t == 1] == 1), R_High=np.mean(p[t == 2] == 2)))
        print(rows[-1])
        pd.DataFrame(rows).to_csv(sens_path, index=False)
if os.path.exists(sens_path):
    S = pd.read_csv(sens_path)
    display(S.groupby("algo")[["MacroF1", "BalAcc", "R_High"]].agg(["min", "median", "max"]).round(3))

In [ ]:
# SEL 16 — (opsional) Cek degenerasi Pers. 3 pada dataset KEEL
# Unggah MC-NRO-master.zip (repositori penulis) bila ingin menjalankan; kalau tidak, biarkan False.
RUN_KEEL = False
if RUN_KEEL:
    import zipfile, glob
    from google.colab import files
    z = files.upload(); zipfile.ZipFile(list(z.keys())[0]).extractall("/content/keel")
    root = glob.glob("/content/keel/**/data/folds", recursive=True)[0]
    from mcnro import MCNRO, ANRO
    from sklearn.tree import DecisionTreeClassifier
    rows = []
    for ds in ["lymphography", "car", "hayes-roth", "balance", "zoo", "glass", "newthyroid"]:
        acc = {"none": [], "MC-NRO": [], "A-NRO": []}; sh = {"MC-NRO": [], "A-NRO": []}; uq = {"MC-NRO": [], "A-NRO": []}
        for pt in range(1, 6):
            for f in (1, 2):
                tr = pd.read_csv(f"{root}/{ds}/{ds}.{pt}.{f}.train.csv").values
                te = pd.read_csv(f"{root}/{ds}/{ds}.{pt}.{f}.test.csv").values
                Xt, yt = tr[:, :-1].astype(float), tr[:, -1].astype(int)
                Xs, ys = te[:, :-1].astype(float), te[:, -1].astype(int)
                acc["none"].append(balanced_accuracy_score(ys, DecisionTreeClassifier(random_state=0).fit(Xt, yt).predict(Xs)))
                for nm, cls in [("MC-NRO", MCNRO), ("A-NRO", ANRO)]:
                    s = cls(k1=5, k2=5, eps=0.5, n_candidates=500, scale=False, random_state=pt * 10 + f)
                    Xr, yr = s.fit_resample(Xt, yt)
                    sh[nm].append(sum(d["n_short"] for d in s.log_) / max(sum(d["n_min"] for d in s.log_), 1))
                    S_ = Xr[s.synthetic_mask_]; uq[nm].append(len(np.unique(S_.round(8), axis=0)) / max(len(S_), 1))
                    acc[nm].append(balanced_accuracy_score(ys, DecisionTreeClassifier(random_state=0).fit(Xr, yr).predict(Xs)))
        rows.append(dict(dataset=ds, short_seed_rate_MCNRO=np.mean(sh["MC-NRO"]), short_seed_rate_ANRO=np.mean(sh["A-NRO"]),
                         unique_synth_ratio_MCNRO=np.mean(uq["MC-NRO"]), unique_synth_ratio_ANRO=np.mean(uq["A-NRO"]),
                         BA_none=np.mean(acc["none"]), BA_MCNRO=np.mean(acc["MC-NRO"]), BA_ANRO=np.mean(acc["A-NRO"])))
        print(rows[-1])
    pd.DataFrame(rows).to_csv(f"{RESULTS_DIR}/keel_degeneracy.csv", index=False)

In [ ]:
# SEL 17 — Ekspor Excel + arsip hasil mentah, lalu unduh
import shutil, os, sys, sklearn, imblearn, numpy, pandas
import pandas as pd
T = A.build_tables(RESULTS_DIR, B=2000)
env = pd.DataFrame({"Komponen": ["python", "numpy", "pandas", "scikit-learn", "imbalanced-learn"],
                    "Versi": [sys.version.split()[0], numpy.__version__, pandas.__version__,
                              sklearn.__version__, imblearn.__version__]})
xlsx = f"{RESULTS_DIR}/ANRO_results_final.xlsx"
with pd.ExcelWriter(xlsx) as w:
    env.to_excel(w, sheet_name="environment", index=False)
    for k, v in T.items():
        v.round(4).to_excel(w, sheet_name=k[:31], index=False)
    for extra in ["sensitivity.csv", "keel_degeneracy.csv"]:
        p = f"{RESULTS_DIR}/{extra}"
        if os.path.exists(p):
            pd.read_csv(p).round(4).to_excel(w, sheet_name=extra[:-4], index=False)
shutil.make_archive("/content/ANRO_raw_results", "zip", RESULTS_DIR)
from google.colab import files
files.download(xlsx)
files.download("/content/ANRO_raw_results.zip")
print("✓ selesai — kirim ANRO_results_final.xlsx dan ANRO_raw_results.zip")